# Task 1: Data Exploration and Enrichment
## Ethiopia Financial Inclusion Forecasting

This notebook explores the starter dataset (`ethiopia_fi_unified_data.csv`) and
`reference_codes.csv`, understands the unified schema, and documents a schema
deviation found during exploration before moving into enrichment.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_csv('../data/raw/ethiopia_fi_unified_data.csv')
ref = pd.read_csv('../data/raw/reference_codes.csv')

print(f"Main dataset shape: {df.shape}")
print(f"Reference codes shape: {ref.shape}")
df.head()


Main dataset shape: (43, 34)
Reference codes shape: (71, 4)


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaN,NaN,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaN,NaN,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


## 1. Schema Overview

In [2]:
print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")


Columns:
 1. record_id
 2. record_type
 3. category
 4. pillar
 5. indicator
 6. indicator_code
 7. indicator_direction
 8. value_numeric
 9. value_text
10. value_type
11. unit
12. observation_date
13. period_start
14. period_end
15. fiscal_year
16. gender
17. location
18. region
19. source_name
20. source_type
21. source_url
22. confidence
23. related_indicator
24. relationship_type
25. impact_direction
26. impact_magnitude
27. impact_estimate
28. lag_months
29. evidence_basis
30. comparable_country
31. collected_by
32. collection_date
33. original_text
34. notes


In [3]:
df.dtypes


record_id                  str
record_type                str
category                   str
pillar                     str
indicator                  str
indicator_code             str
indicator_direction        str
value_numeric          float64
value_text                 str
value_type                 str
unit                       str
observation_date           str
period_start               str
period_end                 str
fiscal_year                str
gender                     str
location                   str
region                 float64
source_name                str
source_type                str
source_url                 str
confidence                 str
related_indicator      float64
relationship_type      float64
impact_direction       float64
impact_magnitude       float64
impact_estimate        float64
lag_months             float64
evidence_basis         float64
comparable_country         str
collected_by               str
collection_date            str
original

## 2. Record Counts by record_type

The brief states the dataset should contain 30 observations, 10 events, 14 impact_links,
and 3 targets. Let's verify.


In [4]:
record_type_counts = df['record_type'].value_counts()
print(record_type_counts)


record_type
observation    30
event          10
target          3
Name: count, dtype: int64


### Schema Deviation Found

The dataset as provided contains **0 `impact_link` records**, not the 14 described in the
brief. There is also **no `parent_id` column** in the schema at all - the brief's instructions
reference linking impact_links to events "via parent_id", but this field does not exist.

**Interpretation:** Task 1's instruction to "Enrich the Dataset... Additional impact_links:
Can you identify other relationships between events and indicators that should be modeled?"
suggests that building the impact_link records is an intentional enrichment exercise for the
trainee, not a data entry omission. This is documented in `data/data_enrichment_log.md` and
addressed in the Enrichment section below, where we construct impact_link rows referencing
events by their `indicator_code`-style event code (e.g. `EVT_TELEBIRR`) via a new
`related_event_code` field, since no `parent_id` column exists to reuse.


## 3. Counts by pillar, source_type, and confidence

In [5]:
print("By pillar:")
print(df['pillar'].value_counts(dropna=False))
print()
print("By source_type:")
print(df['source_type'].value_counts(dropna=False))
print()
print("By confidence:")
print(df['confidence'].value_counts(dropna=False))


By pillar:
pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

By source_type:
source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64

By confidence:
confidence
high      40
medium     3
Name: count, dtype: int64


Note: `event` records have a blank `pillar` by design (per the brief: "Events are categorized
by type... but are NOT pre-assigned to pillars. Their effects on specific indicators are
captured through impact_link records. This keeps the data unbiased.")


## 4. Temporal Range of Observations

In [6]:
obs = df[df['record_type'] == 'observation'].copy()
obs['observation_date'] = pd.to_datetime(obs['observation_date'])

print(f"Earliest observation: {obs['observation_date'].min()}")
print(f"Latest observation: {obs['observation_date'].max()}")
print()
print("Observations per year:")
print(obs['observation_date'].dt.year.value_counts().sort_index())


Earliest observation: 2014-12-31 00:00:00
Latest observation: 2025-12-31 00:00:00

Observations per year:
observation_date
2014     1
2017     1
2021     5
2023     1
2024    11
2025    11
Name: count, dtype: int64


## 5. Unique Indicators and Coverage

In [7]:
indicator_coverage = obs.groupby(['indicator_code', 'indicator']).agg(
    n_observations=('value_numeric', 'count'),
    years=('observation_date', lambda x: sorted(x.dt.year.unique().tolist())),
    pillar=('pillar', 'first')
).reset_index()

indicator_coverage


,indicator_code,indicator,n_observations,years,pillar
0,ACC_4G_COV,4G Population Coverage,2,"[2023, 2025]",ACCESS
1,ACC_FAYDA,Fayda Digital ID Enrollment,3,"[2024, 2025]",ACCESS
2,ACC_MM_ACCOUNT,Mobile Money Account Rate,2,"[2021, 2024]",ACCESS
3,ACC_MOBILE_PEN,Mobile Subscription Penetration,1,[2025],ACCESS
4,ACC_OWNERSHIP,Account Ownership Rate,6,"[2014, 2017, 2021, 2024]",ACCESS
5,AFF_DATA_INCOME,Data Affordability Index,1,[2024],AFFORDABILITY
6,GEN_GAP_ACC,Account Ownership Gender Gap,2,"[2021, 2024]",GENDER
7,GEN_GAP_MOBILE,Mobile Phone Gender Gap,1,[2024],GENDER
8,GEN_MM_SHARE,Female Mobile Money Account Share,1,[2024],GENDER
9,USG_ACTIVE_RATE,Mobile Money Activity Rate,1,[2024],USAGE


## 6. Events Catalogued

In [8]:
events = df[df['record_type'] == 'event'].copy()
events[['record_id', 'indicator_code', 'indicator', 'category', 'observation_date', 'source_name']]


,record_id,indicator_code,indicator,category,observation_date,source_name
33,EVT_0001,EVT_TELEBIRR,Telebirr Launch,product_launch,2021-05-17,Ethio Telecom
34,EVT_0002,EVT_SAFARICOM,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01,News
35,EVT_0003,EVT_MPESA,M-Pesa Ethiopia Launch,product_launch,2023-08-01,Safaricom
36,EVT_0004,EVT_FAYDA,Fayda Digital ID Program Rollout,infrastructure,2024-01-01,NIDP
37,EVT_0005,EVT_FX_REFORM,Foreign Exchange Liberalization,policy,2024-07-29,NBE
38,EVT_0006,EVT_CROSSOVER,P2P Transaction Count Surpasses ATM,milestone,2024-10-01,EthSwitch
39,EVT_0007,EVT_MPESA_INTEROP,M-Pesa EthSwitch Integration,partnership,2025-10-27,EthSwitch
40,EVT_0008,EVT_ETHIOPAY,EthioPay Instant Payment System Launch,infrastructure,2025-12-18,NBE/EthSwitch
41,EVT_0009,EVT_NFIS2,NFIS-II Strategy Launch,policy,2021-09-01,NBE
42,EVT_0010,EVT_SAFCOM_PRICE,Safaricom Ethiopia Price Increase,pricing,2025-12-15,News


## 7. Targets

In [9]:
targets = df[df['record_type'] == 'target'].copy()
targets[['record_id', 'indicator', 'indicator_code', 'value_numeric', 'observation_date', 'notes']]


,record_id,indicator,indicator_code,value_numeric,observation_date,notes
30,REC_0031,Account Ownership Rate,ACC_OWNERSHIP,70.0,2025-12-31,NaN
31,REC_0032,Fayda Digital ID Enrollment,ACC_FAYDA,90000000.0,2028-12-31,NaN
32,REC_0033,Female Mobile Money Account Share,GEN_MM_SHARE,50.0,2030-12-31,NaN


## 8. Reference Codes Overview

In [10]:
ref.groupby('field')['code'].apply(list)


field
category               [product_launch, market_entry, market_exit, po...
confidence                                [high, medium, low, estimated]
evidence_basis              [empirical, literature, theoretical, expert]
gender                                               [all, male, female]
impact_direction                  [increase, decrease, stabilize, mixed]
impact_magnitude                         [high, medium, low, negligible]
indicator_direction               [higher_better, lower_better, neutral]
location                                        [national, urban, rural]
pillar                 [ACCESS, USAGE, QUALITY, AFFORDABILITY, TRUST,...
record_type            [observation, event, impact_link, target, base...
relationship_type             [direct, indirect, enabling, constraining]
source_type            [survey, operator, regulator, policy, news, re...
value_type             [percentage, count, currency_etb, currency_usd...
Name: code, dtype: object